In [19]:
%pip install -r requirements.txt
import time
import pandas as pd
from datasets import load_dataset
from elasticsearch import Elasticsearch, helpers
import requests
from ipywidgets import Text, Button, HBox, VBox, Output, Layout, HTML
from IPython.display import display
from __future__ import annotations

from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import math
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
ds = load_dataset("calmgoose/amazon-product-data-2020")
streaming_ds = load_dataset("calmgoose/amazon-product-data-2020", split="train", streaming=True)
streaming_ds,next(iter(streaming_ds))




(IterableDataset({
     features: ['Uniq Id', 'Product Name', 'Category', 'Upc Ean Code', 'Selling Price', 'Model Number', 'About Product', 'Product Specification', 'Technical Details', 'Shipping Weight', 'Product Dimensions', 'Image', 'Variants', 'Product Url', 'Is Amazon Seller'],
     num_shards: 1
 }),
 {'Uniq Id': '4c69b61db1fc16e7013b43fc926e502d',
  'Product Name': 'DB Longboards CoreFlex Crossbow 41" Bamboo Fiberglass Longboard Complete',
  'Category': 'Sports & Outdoors | Outdoor Recreation | Skates, Skateboards & Scooters | Skateboarding | Standard Skateboards & Longboards | Longboards',
  'Upc Ean Code': None,
  'Selling Price': '$237.68',
  'Model Number': None,
  'About Product': "Make sure this fits by entering your model number. | RESPONSIVE FLEX: The Crossbow features a bamboo core encased in triaxial fiberglass and HD plastic for a responsive flex pattern that’s second to none. Pumping & carving have never been so satisfying! Flex 2 is recommended for people 120 to 170

In [21]:
#Start Elasticsearch server
ES_URL = "http://127.0.0.1:9200"
!docker run -d --name es-dev -p 9200:9200 -e "discovery.type=single-node" -e "xpack.security.enabled=false" docker.elastic.co/elasticsearch/elasticsearch:8.14.0
for i in range(60):
    try:
        r = requests.get(ES_URL, timeout=2)
        if r.ok:
            print("Elasticsearch is up:", r.json().get("cluster_name"), r.json().get("version", {}).get("number"))
            break
    except Exception as e:
        time.sleep(2)
else:
    raise RuntimeError("Elasticsearch did not start within the expected time")


Elasticsearch is up: docker-cluster 8.14.0


docker: Error response from daemon: Conflict. The container name "/es-dev" is already in use by container "15d63b7e35e506c2273abdafa79d629d377a999d9dfe89d7756341afbabc81cc". You have to remove (or rename) that container to be able to reuse that name.

Run 'docker run --help' for more information


In [22]:
# Connect explicitly to the Docker container endpoint started in the previous cell
# Use options suitable for local, insecure HTTP and add robust retry behavior
es = Elasticsearch(
    ES_URL,
    request_timeout=30,
    retry_on_timeout=True,
    verify_certs=False  # local HTTP container (no TLS)
)

# Use a direct info() call (more reliable than ping) and allow more time/backoff for the container to be ready
last_err = None
for i in range(10):  # up to ~120s with 2s sleeps
    try:
        info = es.info()
        print("Connected to Elasticsearch container:", info.get("cluster_name"), info.get("version", {}).get("number"))
        break
    except Exception as e:
        last_err = e
        time.sleep(2)
else:
    raise RuntimeError(
        f"Could not connect to Elasticsearch container at {ES_URL}. Ensure Docker container 'es-dev' is running and port 9200 is published.\n"
        f"Last error: {type(last_err).__name__}: {last_err}"
    )


Connected to Elasticsearch container: docker-cluster 8.14.0


In [23]:
INDEX = "amazon_products_2020"

# Stream to avoid downloading everything at once
streaming_ds = load_dataset("calmgoose/amazon-product-data-2020", split="train", streaming=True)

def to_action(ex):
    doc = dict(ex)
    # Try common ID fields; fall back to None (ES will auto-generate)
    _id = doc.get("asin") or doc.get("uniq_id")
    return {
        "_index": INDEX,
        "_id": _id,
        "_source": doc
    }

batch = []
BATCH_SIZE = 2000
count = 0

for ex in streaming_ds:
    batch.append(to_action(ex))
    if len(batch) >= BATCH_SIZE:
        helpers.bulk(es, batch)
        count += len(batch)
        batch.clear()

# Flush remainder
if batch:
    helpers.bulk(es, batch)
    count += len(batch)

print(f"Indexed {count} documents")

Indexed 10002 documents


In [24]:
# Important fields for candidate_text construction
DEFAULT_FIELDS: Tuple[str, ...] = (
    "Product Name",
    "About Product",
    "Product Specification",
    "Technical Details",
)


def _first_non_empty(d: Dict[str, Any], keys: Sequence[str]) -> Optional[str]:
    for k in keys:
        v = d.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None


def build_candidate_text(
    source: Dict[str, Any],
    fields: Sequence[str] = DEFAULT_FIELDS,
    max_chars_per_field: int = 300,
    max_total_chars: int = 1200,
    sep: str = " | ",
) -> str:
    """
    Build a concise candidate text from ES _source using important fields.

    - Picks a title from common keys.
    - Appends selected fields, each truncated to max_chars_per_field.
    - Ensures total length stays reasonably small for cross-encoders.
    """
    parts: List[str] = []

    # Prefer a title-like field
    title = _first_non_empty(source, ["Product Name", "product_name", "title", "name"])
    if title:
        parts.append(title.strip())

    # Add important fields
    total_len = sum(len(p) for p in parts)
    for f in fields:
        v = source.get(f)
        if not isinstance(v, str):
            continue
        s = " ".join(v.split())  # squash whitespace
        if not s:
            continue
        if max_chars_per_field:
            s = s[: max_chars_per_field].rstrip()
        parts.append(f"{f}: {s}")
        total_len += len(parts[-1])
        if max_total_chars and total_len >= max_total_chars:
            break

    # Fallback: include some other possibly relevant text fields if nothing else
    if len(parts) == 0:
        for k, v in source.items():
            if isinstance(v, str) and v.strip():
                parts.append(" ".join(v.split())[: max_chars_per_field])
                break

    text = sep.join(parts)
    if max_total_chars and len(text) > max_total_chars:
        text = text[: max_total_chars].rstrip()
    return text


class JinaReranker:
    """Reranker using jinaai/jina-reranker-v2-base-multilingual.

    Scores (query, doc) pairs with a relevance score. Higher is more relevant.
    """

    def __init__(
        self,
        model_name: str = "jinaai/jina-reranker-v2-base-multilingual",
        device: Optional[str] = None,
        batch_size: int = 16,
        max_length: int = 512,
        use_fp16: bool = True,
    ) -> None:
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length

        if device is None:
            if torch.cuda.is_available():
                device = "cuda"
            else:
                device = "cpu"
        self.device = torch.device(device)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)
        self.model.to(self.device)
        self.model.eval()

        # Determine positive label index for classification heads if present
        self.pos_label_idx: Optional[int] = None
        id2label = getattr(self.model.config, "id2label", None)
        if isinstance(id2label, dict) and len(id2label) >= 2:
            # Try to find a label that looks like the positive class
            for idx_str, label in id2label.items():
                try:
                    idx = int(idx_str)
                except Exception:
                    # In some configs keys are ints already
                    idx = int(idx_str) if isinstance(idx_str, int) else None
                if idx is None:
                    continue
                if str(label).lower() in {"relevant", "entailment", "pos", "positive"}:
                    self.pos_label_idx = idx
                    break
            if self.pos_label_idx is None:
                # Fall back to the last index
                self.pos_label_idx = max(int(i) for i in id2label.keys())

        self.use_fp16 = use_fp16 and (self.device.type == "cuda")

    @torch.inference_mode()
    def score(self, query: str, docs: Sequence[str]) -> List[float]:
        scores: List[float] = []
        if not docs:
            return scores

        for i in range(0, len(docs), self.batch_size):
            batch_docs = docs[i : i + self.batch_size]
            enc = self.tokenizer(
                [query] * len(batch_docs),
                list(batch_docs),
                truncation=True,
                padding=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}

            if self.use_fp16:
                with torch.autocast(device_type=self.device.type, dtype=torch.float16):
                    logits = self.model(**enc).logits
            else:
                logits = self.model(**enc).logits

            if logits.shape[-1] == 1:
                batch_scores = logits.squeeze(-1).detach().float().tolist()
            else:
                # Assume classification; use softmax prob of positive label
                probs = torch.softmax(logits, dim=-1)
                pos_idx = self.pos_label_idx if self.pos_label_idx is not None else logits.shape[-1] - 1
                batch_scores = probs[:, pos_idx].detach().float().tolist()
            scores.extend(batch_scores)
        return scores

    def rank_hits(
        self,
        query: str,
        hits: Sequence[Dict[str, Any]],
        *,
        fields: Sequence[str] = DEFAULT_FIELDS,
        top_n: Optional[int] = None,
        blend_alpha: Optional[float] = None,
        max_chars_per_field: int = 300,
        max_total_chars: int = 1200,
    ) -> List[Dict[str, Any]]:
        """
        Rerank ES hits (list of hits as in resp["hits"]["hits"]). Returns a new sorted list of hits.

        - If blend_alpha is provided in [0,1], blend normalized ES _score with reranker score.
        - Otherwise, sort by reranker score alone.
        """
        if not hits:
            return []

        # Build candidate texts
        cand_texts: List[str] = []
        for h in hits:
            src = h.get("_source", {}) or {}
            cand_texts.append(
                build_candidate_text(
                    src,
                    fields=fields,
                    max_chars_per_field=max_chars_per_field,
                    max_total_chars=max_total_chars,
                )
            )

        rr_scores = self.score(query, cand_texts)

        # Attach scores
        enriched: List[Dict[str, Any]] = []
        for h, rr in zip(hits, rr_scores):
            h2 = dict(h)  # shallow copy
            h2["_rerank_score"] = float(rr)
            enriched.append(h2)

        # Blending with ES _score if requested
        if blend_alpha is not None:
            a = float(max(0.0, min(1.0, blend_alpha)))
            es_scores = [float(h.get("_score", 0.0)) for h in enriched]
            es_norm = _min_max_normalize(es_scores)
            rr_norm = _min_max_normalize([h["_rerank_score"] for h in enriched])
            for i, h in enumerate(enriched):
                h["_final_score"] = a * rr_norm[i] + (1.0 - a) * es_norm[i]
            key = "_final_score"
        else:
            key = "_rerank_score"

        enriched.sort(key=lambda x: x.get(key, -math.inf), reverse=True)
        if top_n is not None:
            enriched = enriched[:top_n]
        return enriched


def _min_max_normalize(values: Sequence[float]) -> List[float]:
    if not values:
        return []
    vmin = min(values)
    vmax = max(values)
    if vmax <= vmin:
        return [0.0 for _ in values]
    return [(v - vmin) / (vmax - vmin) for v in values]


def retrieve_and_rerank(
    es_client: Any,
    index: str,
    query_text: str,
    search_fields: Sequence[str],
    *,
    top_k: int = 100,
    top_n: int = 20,
    highlight_fields: Optional[Sequence[str]] = None,
    blend_alpha: Optional[float] = None,
    model: Optional[JinaReranker] = None,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Executes ES retrieval, then reranks.

    Returns (reranked_hits, raw_hits).
    """
    hl = None
    if highlight_fields:
        hl = {"fields": {f: {} for f in highlight_fields}}

    resp = es_client.search(
        index=index,
        query={"multi_match": {"query": query_text, "fields": list(search_fields)}},
        highlight=hl,
        size=top_k,
    )
    hits = resp.get("hits", {}).get("hits", [])

    reranker = model or JinaReranker()
    ranked = reranker.rank_hits(
        query_text,
        hits,
        fields=DEFAULT_FIELDS,
        top_n=top_n,
        blend_alpha=blend_alpha,
    )
    return ranked, hits


In [25]:
search_fields = [
    'Uniq Id',
    'Product Name',
    'Category',
    'Upc Ean Code',
    'Selling Price',
    'Model Number',
    'About Product',
    'Product Specification',
    'Technical Details',
    'Shipping Weight',
    'Product Dimensions',
    'Image',
    'Variants',
    'Product Url',
    'Is Amazon Seller'
]

css = HTML("""
<style>
.custom-search .widget-text input {
  font-size: 16px;
  padding: 10px 14px;
}
.custom-search .widget-button {
  font-size: 16px;
  padding: 8px 14px;
}
/* Bigger label for the Text description ("Query:") */
.custom-search .widget-label {
  font-size: 18px;
}
</style>
""")

q = Text(
    value="wireless noise cancelling headphones",
    placeholder="Type your search…",
    description="Query:",
    disabled=False,
    layout=Layout(width="600px", height="40px"),
    style={"description_width": "90px"}  # make room for the bigger label
)
q.continuous_update = False

btn = Button(
    description="Search",
    button_style="primary",  # 'primary', 'success', 'info', 'warning', 'danger' or ''
    layout=Layout(width="600px", height="40px")  # match Text width
)

out = Output()


def run_search(query_text: str):
    with out:
        out.clear_output()
        if not query_text.strip():
            print("Please enter a query.")
            return
        # Execute initial retrieval from Elasticsearch (top_k=50)
        resp = es.search(
            index=INDEX,
            query={"multi_match": {"query": query_text, "fields": search_fields}},
            highlight={
                "fields": {
                    "Product Name": {},
                    "About Product": {},
                    "Product Specification": {},
                    "Technical Details": {},
                    "Category": {}
                }
            },
            size=50
        )
        hits = resp.get("hits", {}).get("hits", [])
        # Save for reranking in the next cell
        globals()["LAST_QUERY"] = query_text
        globals()["LAST_ES_HITS"] = hits

        # Show initial ES-ranked results (only first 5)
        for hit in hits[:5]:
            src = hit.get("_source", {})
            title = src.get("Product Name") or src.get("product_name") or src.get("title") or "(no title)"
            print(f"{hit.get('_score', 0.0):.4f}  {title}")
            if "highlight" in hit:
                for field, frags in hit["highlight"].items():
                    print("  ", field, "=>", " ... ".join(frags))
        print("\nTip: Run the next cell to rerank these results with the Jina reranker.")


def on_text_change(change):
    if change.get("name") == "value":
        run_search(q.value)


def on_click(b):
    run_search(q.value)


q.observe(on_text_change, names="value")
btn.on_click(on_click)

# Center both controls and stack them vertically
controls = VBox([q, btn], layout=Layout(align_items="center", width="100%", gap="10px"))

# Inject CSS and apply a class wrapper for scoping
container = VBox([css, controls, out])
container.add_class("custom-search")

display(container)


In [27]:
# Rerank the results from the previous cell using the Jina multilingual reranker
# - Expects LAST_QUERY and LAST_ES_HITS to be set by the search cell above.
# - Prints results ordered by cross-encoder relevance score.
_prev_query = globals().get("LAST_QUERY", None)
_prev_hits = globals().get("LAST_ES_HITS", None)

if _prev_query is None or _prev_hits is None:
    print("No previous search results found. Please run the search cell first.")
elif not _prev_hits:
    print("No hits to rerank. Try a different query in the search cell.")
else:
    print(f"Reranking {_prev_hits and len(_prev_hits) or 0} hits for query: {_prev_query!r}")
    try:
        reranker = JinaReranker()  # jinaai/jina-reranker-v2-base-multilingual
        ranked_hits = reranker.rank_hits(
            _prev_query,
            _prev_hits,
            fields=DEFAULT_FIELDS,
            top_n=1,        # keep all; set an int to truncate
            blend_alpha=None,  # set e.g. 0.3 to blend with ES _score
        )
        for hit in ranked_hits:
            src = hit.get("_source", {})
            title = src.get("Product Name") or src.get("product_name") or src.get("title") or "(no title)"
            score_to_show = hit.get("_rerank_score", hit.get("_score", 0.0))
            print(f"{score_to_show:.4f}  {title}")
            if "highlight" in hit:
                for field, frags in hit["highlight"].items():
                    print("  ", field, "=>", " ... ".join(frags))
    except Exception as e:
        print("Warning: Reranking failed:", e)

Reranking 36 hits for query: 'rice cooker'
-2.0781  DJECO Gaby's Cooker Wooden Role Play Set, Multicolor
   Product Name => DJECO Gaby's <em>Cooker</em> Wooden Role Play Set, Multicolor
